In [ ]:
"""
Temporal Fusion Transformer - Flood Forecasting
Predicts streamflow 24 hours ahead using the top30 high flood-severity sites.
Logs all metrics, charts, and model artifacts to Weights & Biases.
"""
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import wandb
from keras.callbacks import EarlyStopping

from src.preprocessing.preprocessing import processor
from src.models.tft import TemporalFusionTransformerModel

STATIC_FEATURES = [
    "longitude", "latitude", "DRAIN_SQKM", "artificial_path_pct",
    "wb5100_ann_mm", "snw_pc_syr", "snow_ice_nlcd06", "barren_nlcd06",
    "mains100_plant", "hga", "hgc", "bulk_density_avg", "elev_max_m", "aspect_deg",
]

DYNAMIC_FEATURES = [
    "streamflow_cfs_mean", "streamflow_cfs_max", "streamflow_cfs_min",
    "gage_height_ft_mean",
    "precipitation_mm",
    "temperature_c",
    "potential_evaporation_mm",
    "specific_humidity_kgkg",
    "shortwave_radiation_wm2",
    "longwave_radiation_wm2",
    "wind_speed_ms",
    "surface_pressure_pa",
    "cape_jkg",
    "convective_precip_fraction",
]

WINDOW_SIZE = 72

In [ ]:
# ── Hyperparameters tracked by W&B ────────────────────────────────────────────
wandb_config = {
    # Architecture
    "num_static_features": len(STATIC_FEATURES),
    "d_model": 64,
    "num_heads": 4,
    "lstm_units": 64,
    "dropout_rate": 0.1,
    # Training
    "learning_rate": 1e-3,
    "under_predict_penalty": 2.0,
    "epochs": 50,
    "batch_size": 512,
    "early_stopping_patience": 5,
    # Data
    "window_size": WINDOW_SIZE,
    "train_split": 0.8,
    "val_split": 0.9,
    "lag_window": 1,
    "frequency": "hourly",
    "split_time_days": 30,
    "dataset": "flood-dataset-top30",
    "target": "streamflow_cfs_target_24h",
}

run = wandb.init(
    project="flood-forecasting",
    name="tft",
    config=wandb_config,
    tags=["tft", "temporal-fusion-transformer", "top30"],
)

cfg = wandb.config
print(f"W&B run: {run.name}  |  id: {run.id}")

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
config = {
    "input_cols": DYNAMIC_FEATURES + STATIC_FEATURES,
    "static_cols": STATIC_FEATURES,
    "target": cfg.target,
    "train_split": cfg.train_split,
    "val_split": cfg.val_split,
    "file_path": cfg.dataset,
    "file_name": "flood_model_top30",
    "table": "wandb.flood_model_top30",
    "lag_window": cfg.lag_window,
    "frequency": cfg.frequency,
    "split_time_days": cfg.split_time_days,
    "site_scaling": False,
}

pcr = processor(config)
pcr.pull_wandb()
print(pcr.df["site_id"].unique())
print(pcr.df.shape)
train_X, val_X, test_X, train_y, val_y, test_y = pcr.return_outputs()

In [ ]:
# ── Lazy sequence dataset ─────────────────────────────────────────────────────
# PyTorch Dataset that generates sliding windows on-the-fly per site.
# Only the per-site 2-D arrays and a flat index map are held in memory —
# the full 3-D window tensor is never materialised.

class SequenceDataset(Dataset):
    """Map-style PyTorch dataset of (window, target) pairs."""

    def __init__(self, X_df, y_df, window_size: int):
        feature_cols = [c for c in X_df.columns if c not in ("site_id", "observation_hour")]
        target_col = y_df.columns[0]

        self.window_size = window_size
        self.site_data: list[tuple[np.ndarray, np.ndarray]] = []
        self.index_map: list[tuple[int, int]] = []

        for site in sorted(X_df["site_id"].unique().to_list()):
            mask = X_df["site_id"] == site
            sx = X_df.filter(mask).select(feature_cols).to_numpy().astype("float32")
            sy = y_df.filter(mask)[target_col].to_numpy().astype("float32")
            if len(sx) > window_size:
                site_idx = len(self.site_data)
                self.site_data.append((sx, sy))
                for i in range(len(sx) - window_size):
                    self.index_map.append((site_idx, i))

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        site_idx, start = self.index_map[idx]
        sx, sy = self.site_data[site_idx]
        x = torch.from_numpy(sx[start : start + self.window_size])
        y = torch.tensor(sy[start + self.window_size - 1])
        return x, y


print("Building datasets...")
train_dataset = SequenceDataset(train_X, train_y, WINDOW_SIZE)
val_dataset   = SequenceDataset(val_X,   val_y,   WINDOW_SIZE)

use_cuda = torch.cuda.is_available()
train_ds = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, pin_memory=use_cuda)
val_ds   = DataLoader(val_dataset,   batch_size=cfg.batch_size, shuffle=False, pin_memory=use_cuda)
print(f"Datasets ready: {len(train_dataset):,} train / {len(val_dataset):,} val sequences")

feature_cols = [c for c in train_X.columns if c not in ("site_id", "observation_hour")]
n_features = len(feature_cols)
print(f"Window: {WINDOW_SIZE} steps  |  Features: {n_features}")

wandb.log({
    "data/n_features": n_features,
    "data/n_train_sites": train_X["site_id"].n_unique(),
})

In [ ]:
import keras
print("Keras backend:", keras.backend.backend())
print("Torch CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# Check where Keras is placing tensors
sample_x, sample_y = next(iter(train_ds))
print("Batch device:", sample_x.device)

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA compiled:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<not set>"))
print("PATH has cuda:", any("cuda" in p.lower() for p in os.environ.get("PATH", "").split(";")))

In [ ]:
# ── Build model ───────────────────────────────────────────────────────────────
tft = TemporalFusionTransformerModel(
    num_static_features=cfg.num_static_features,
    d_model=cfg.d_model,
    num_heads=cfg.num_heads,
    lstm_units=cfg.lstm_units,
    dropout_rate=cfg.dropout_rate,
    under_predict_penalty=cfg.under_predict_penalty,
    learning_rate=cfg.learning_rate,
)

tft.build(input_shape=(WINDOW_SIZE, n_features))
tft.summary()

total_params = tft.model.count_params()
wandb.log({"model/total_params": total_params})
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=cfg.early_stopping_patience,
    restore_best_weights=True,
)

# WandbMetricsLogger logs train/val loss and MAE each epoch automatically
wandb_logger = wandb.keras.WandbMetricsLogger(log_freq="epoch")

history = tft.fit(
    train_ds,
    val_ds=val_ds,
    epochs=cfg.epochs,
    callbacks=[early_stopping, wandb_logger],
)

epochs_trained = len(history.history["loss"])
best_val_loss  = min(history.history["val_loss"])
best_val_mae   = min(history.history["val_mae"])

wandb.log({
    "train/epochs_trained": epochs_trained,
    "train/best_val_loss": best_val_loss,
    "train/best_val_mae": best_val_mae,
})
print(f"\nTraining complete: {epochs_trained} epochs  |  best val loss: {best_val_loss:.4f}  |  best val MAE: {best_val_mae:.4f}")

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
tft.plot_training_history()

epochs = list(range(1, epochs_trained + 1))

loss_table = wandb.Table(
    columns=["epoch", "train_loss", "val_loss"],
    data=[[e, tl, vl] for e, tl, vl in zip(
        epochs, history.history["loss"], history.history["val_loss"]
    )],
)
mae_table = wandb.Table(
    columns=["epoch", "train_mae", "val_mae"],
    data=[[e, tm, vm] for e, tm, vm in zip(
        epochs, history.history["mae"], history.history["val_mae"]
    )],
)

wandb.log({
    "charts/loss_curve": wandb.plot.line(
        loss_table, x="epoch", y="train_loss", title="Train Loss"
    ),
    "charts/val_loss_curve": wandb.plot.line(
        loss_table, x="epoch", y="val_loss", title="Val Loss"
    ),
    "charts/mae_curve": wandb.plot.line(
        mae_table, x="epoch", y="train_mae", title="Train MAE"
    ),
    "charts/val_mae_curve": wandb.plot.line(
        mae_table, x="epoch", y="val_mae", title="Val MAE"
    ),
})

In [ ]:
# ── Postprocessing: per-site prediction loop ──────────────────────────────────
# Predict site by site so the window array is only (site_sequences, W, F) at a
# time — the largest site is ~9 K sequences, a few hundred MB at most.

pred_scaled_all   = []
actual_scaled_all = []
test_site_ids     = []

target_col   = test_y.columns[0]

for site in sorted(test_X["site_id"].unique().to_list()):
    mask   = test_X["site_id"] == site
    site_X = test_X.filter(mask).select(feature_cols).to_numpy().astype("float32")
    site_y = test_y.filter(mask)[target_col].to_numpy().astype("float32")

    if len(site_X) <= WINDOW_SIZE:
        continue

    n_windows = len(site_X) - WINDOW_SIZE
    # Materialise windows for this site only
    idx      = np.arange(WINDOW_SIZE)[None, :] + np.arange(n_windows)[:, None]
    windows  = site_X[idx]                           # (n_windows, W, F)
    targets  = site_y[WINDOW_SIZE - 1 : WINDOW_SIZE - 1 + n_windows]

    preds = tft.predict(windows, verbose=0).flatten()
    pred_scaled_all.extend(preds)
    actual_scaled_all.extend(targets)
    test_site_ids.extend([site] * n_windows)

pred_scaled_all   = np.array(pred_scaled_all,   dtype="float32")
actual_scaled_all = np.array(actual_scaled_all, dtype="float32")
test_site_ids     = np.array(test_site_ids)

# Inverse-transform back to CFS
pred_cfs = pcr.target_scaler.inverse_transform(
    torch.tensor(pred_scaled_all).reshape(-1, 1)
).numpy().flatten()

actual_cfs = pcr.target_scaler.inverse_transform(
    torch.tensor(actual_scaled_all).reshape(-1, 1)
).numpy().flatten()

mae_cfs  = float(np.abs(pred_cfs - actual_cfs).mean())
rmse_cfs = float(np.sqrt(np.mean((pred_cfs - actual_cfs) ** 2)))
nse      = float(1 - (
    np.sum((actual_cfs - pred_cfs) ** 2) /
    np.sum((actual_cfs - actual_cfs.mean()) ** 2)
))
bias_pct = float((pred_cfs.mean() - actual_cfs.mean()) / actual_cfs.mean() * 100)

print(f"{'Metric':<12} {'Value':>12}")
print("-" * 26)
print(f"{'MAE':<12} {mae_cfs:>10.1f} CFS")
print(f"{'RMSE':<12} {rmse_cfs:>10.1f} CFS")
print(f"{'NSE':<12} {nse:>12.4f}")
print(f"{'Bias':<12} {bias_pct:>10.1f} %")
print(f"\nActual mean:    {actual_cfs.mean():.1f} CFS")
print(f"Predicted mean: {pred_cfs.mean():.1f} CFS")

wandb.log({
    "test/mae_cfs":  mae_cfs,
    "test/rmse_cfs": rmse_cfs,
    "test/nse":      nse,
    "test/bias_pct": bias_pct,
})

In [ ]:
# ── Per-site metrics table ────────────────────────────────────────────────────
import polars as pl

site_rows = []
for site in np.unique(test_site_ids):
    mask = test_site_ids == site
    a = actual_cfs[mask]
    p = pred_cfs[mask]
    site_mae  = float(np.abs(p - a).mean())
    site_rmse = float(np.sqrt(np.mean((p - a) ** 2)))
    site_nse  = float(
        1 - (np.sum((a - p) ** 2) / np.sum((a - a.mean()) ** 2))
        if a.std() > 0 else float("nan")
    )
    site_rows.append({
        "site_id": site,
        "actual_mean_cfs": round(float(a.mean()), 1),
        "predicted_mean_cfs": round(float(p.mean()), 1),
        "mae_cfs": round(site_mae, 1),
        "rmse_cfs": round(site_rmse, 1),
        "nse": round(site_nse, 4),
    })
    print(
        f"Site {site}: actual={a.mean():.1f}  pred={p.mean():.1f}  "
        f"MAE={site_mae:.1f}  NSE={site_nse:.3f}"
    )

site_df = pl.DataFrame(site_rows)

wandb.log({
    "test/per_site_metrics": wandb.Table(
        columns=site_df.columns,
        data=site_df.to_pandas().values.tolist(),
    )
})

In [ ]:
# ── Prediction vs actual time-series chart ────────────────────────────────────
import matplotlib.pyplot as plt

n_plot = 500
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(actual_cfs[:n_plot], label="Actual", alpha=0.8)
ax.plot(pred_cfs[:n_plot],   label="Predicted", alpha=0.8)
ax.set_xlabel("Time step")
ax.set_ylabel("Streamflow (CFS)")
ax.set_title(f"TFT: Predicted vs Actual (first {n_plot} test samples)")
ax.legend()
plt.tight_layout()
wandb.log({"charts/predictions_vs_actual": wandb.Image(fig)})
plt.show()

In [ ]:
# ── Scatter: predicted vs actual ──────────────────────────────────────────────
sample_idx = np.random.choice(len(actual_cfs), size=min(5000, len(actual_cfs)), replace=False)
scatter_table = wandb.Table(
    columns=["actual_cfs", "predicted_cfs"],
    data=[[float(actual_cfs[i]), float(pred_cfs[i])] for i in sample_idx],
)
wandb.log({
    "charts/scatter_pred_vs_actual": wandb.plot.scatter(
        scatter_table, x="actual_cfs", y="predicted_cfs",
        title="Predicted vs Actual Streamflow (CFS)",
    )
})

In [ ]:
# ── Save model and log as W&B artifact ────────────────────────────────────────
saved_path = tft.save_model(path="notebooks/model_training/tft", name="tft_model")
print(f"Model saved to {saved_path}")

artifact = wandb.Artifact(
    name="tft-model",
    type="model",
    description="Temporal Fusion Transformer trained on top-30 flood sites",
    metadata={
        "mae_cfs":  mae_cfs,
        "rmse_cfs": rmse_cfs,
        "nse":      nse,
        **dict(cfg),
    },
)
artifact.add_file(str(saved_path))
run.log_artifact(artifact)
print("Model artifact logged to W&B.")

In [ ]:
wandb.finish()